# Experiment 2: DeepSpeed vs PyTorch Inference Comparison


Compare inference performance on OPT-6.7B (~13GB in fp16),
a model that FITS on A100-40GB in both setups.

This lets us do a fair comparison of:
  - Time to First Token (TTFT)
  - Throughput (tokens/sec)
  - Total generation time
  - Peak GPU memory usage

Expected outcome:
  - PyTorch wins on SPEED (no CPU↔GPU transfer overhead)
  - DeepSpeed wins on MEMORY (parameters offloaded to CPU)
  - This shows the tradeoff: DeepSpeed sacrifices speed for memory savings
  - Combined with Exp 1, the story is: when the model doesn't fit,
    DeepSpeed is the ONLY option

In [ ]:
!pip install deepspeed transformers accelerate matplotlib --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.3 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import gc
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import deepspeed
from transformers.integrations.deepspeed import HfDeepSpeedConfig


## CONFIG

In [ ]:

MODEL_NAME = "facebook/opt-6.7b"  # ~13GB in fp16, fits on A100-40GB
MAX_NEW_TOKENS = 100

TEST_PROMPTS = [
    "Explain the theory of relativity in simple terms:",
    "Write a Python function to sort a list using merge sort:",
    "What are the main differences between CPU and GPU architectures?",
    "Summarize the key events of World War II:",
    "Describe how a neural network learns from data step by step:",
]

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def clear_gpu():
    """Clear GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()


def measure_inference(model, tokenizer, prompt, max_new_tokens, device):
    """Measure inference performance for a single prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    # Measure time to first token
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        first_out = model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize()
    ttft = time.time() - start

    # Measure full generation
    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        full_out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize()
    total_time = time.time() - start

    num_tokens = full_out.shape[1] - input_len

    return {
        "ttft": ttft,
        "total_time": total_time,
        "num_tokens": num_tokens,
        "tokens_per_sec": num_tokens / total_time if total_time > 0 else 0,
    }



## PART A: VANILLA PYTORCH INFERENCE

In [ ]:
print("=" * 70)
print("EXPERIMENT 2: DeepSpeed vs PyTorch Inference Comparison")
print(f"Model: {MODEL_NAME} (fits on A100-40GB in both setups)")
print("=" * 70)

print("\n" + "-" * 50)
print("Part A: Vanilla PyTorch Inference")
print("-" * 50)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_NAME} with vanilla PyTorch...")
model_pytorch = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
model_pytorch.eval()
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Warmup
print("Warming up...")
warmup_inputs = tokenizer("Hello", return_tensors="pt").to("cuda")
with torch.no_grad():
    model_pytorch.generate(**warmup_inputs, max_new_tokens=5, do_sample=False)

# Benchmark
pytorch_results = []
print(f"\nRunning {len(TEST_PROMPTS)} prompts...")

for i, prompt in enumerate(TEST_PROMPTS):
    result = measure_inference(model_pytorch, tokenizer, prompt, MAX_NEW_TOKENS, "cuda")
    pytorch_results.append(result)
    print(f"  Prompt {i+1}: {result['tokens_per_sec']:.1f} tok/s | TTFT: {result['ttft']:.3f}s | Total: {result['total_time']:.2f}s")

pytorch_peak_memory = torch.cuda.max_memory_allocated() / 1e9
pytorch_avg_ttft = sum(r["ttft"] for r in pytorch_results) / len(pytorch_results)
pytorch_avg_throughput = sum(r["tokens_per_sec"] for r in pytorch_results) / len(pytorch_results)
pytorch_avg_total = sum(r["total_time"] for r in pytorch_results) / len(pytorch_results)

print(f"\nPyTorch Averages:")
print(f"  Avg TTFT:        {pytorch_avg_ttft:.3f}s")
print(f"  Avg throughput:  {pytorch_avg_throughput:.1f} tokens/sec")
print(f"  Avg total time:  {pytorch_avg_total:.2f}s")
print(f"  Peak GPU memory: {pytorch_peak_memory:.2f} GB")

# Cleanup
del model_pytorch
clear_gpu()
time.sleep(5)



EXPERIMENT 2: DeepSpeed vs PyTorch Inference Comparison
Model: facebook/opt-6.7b (fits on A100-40GB in both setups)

--------------------------------------------------
Part A: Vanilla PyTorch Inference
--------------------------------------------------


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GPU memory after loading: 13.32 GB
Warming up...

Running 5 prompts...
  Prompt 1: 39.5 tok/s | TTFT: 0.026s | Total: 2.53s
  Prompt 2: 39.8 tok/s | TTFT: 0.025s | Total: 2.51s
  Prompt 3: 39.7 tok/s | TTFT: 0.025s | Total: 2.52s
  Prompt 4: 40.2 tok/s | TTFT: 0.025s | Total: 2.48s
  Prompt 5: 39.9 tok/s | TTFT: 0.025s | Total: 2.50s

PyTorch Averages:
  Avg TTFT:        0.025s
  Avg throughput:  39.8 tokens/sec
  Avg total time:  2.51s
  Peak GPU memory: 13.39 GB


## PART B: DEEPSPEED ZERO INFERENCE

In [ ]:
print("\n" + "-" * 50)
print("Part B: DeepSpeed ZeRO Inference (with CPU offloading)")
print("-" * 50)



os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29501"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

if not torch.distributed.is_initialized():
    torch.distributed.init_process_group(backend="nccl", world_size=1, rank=0)

ds_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_param": {
            "device": "cpu",
            "pin_memory": True,
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
    },
    "train_batch_size": 1,
    "train_micro_batch_size_per_gpu": 1,
    "steps_per_print": 2000,
    "wall_clock_breakdown": False,
}

dschf = HfDeepSpeedConfig(ds_config)

print(f"Loading {MODEL_NAME} with DeepSpeed CPU offloading...")
model_ds = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
)

ds_engine = deepspeed.initialize(
    model=model_ds,
    config_params=ds_config,
)[0]
ds_engine.module.eval()
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Warmup
print("Warming up...")
warmup_inputs = tokenizer("Hello", return_tensors="pt").to(ds_engine.module.device)
with torch.no_grad():
    ds_engine.module.generate(**warmup_inputs, max_new_tokens=5, do_sample=False)

# Benchmark
ds_results = []
print(f"\nRunning {len(TEST_PROMPTS)} prompts...")

for i, prompt in enumerate(TEST_PROMPTS):
    result = measure_inference(ds_engine.module, tokenizer, prompt, MAX_NEW_TOKENS, ds_engine.module.device)
    ds_results.append(result)
    print(f"  Prompt {i+1}: {result['tokens_per_sec']:.1f} tok/s | TTFT: {result['ttft']:.3f}s | Total: {result['total_time']:.2f}s")

ds_peak_memory = torch.cuda.max_memory_allocated() / 1e9
ds_avg_ttft = sum(r["ttft"] for r in ds_results) / len(ds_results)
ds_avg_throughput = sum(r["tokens_per_sec"] for r in ds_results) / len(ds_results)
ds_avg_total = sum(r["total_time"] for r in ds_results) / len(ds_results)

print(f"\nDeepSpeed Averages:")
print(f"  Avg TTFT:        {ds_avg_ttft:.3f}s")
print(f"  Avg throughput:  {ds_avg_throughput:.1f} tokens/sec")
print(f"  Avg total time:  {ds_avg_total:.2f}s")
print(f"  Peak GPU memory: {ds_peak_memory:.2f} GB")



--------------------------------------------------
Part B: DeepSpeed ZeRO Inference (with CPU offloading)
--------------------------------------------------
Loading facebook/opt-6.7b with DeepSpeed CPU offloading...
GPU memory after loading: 0.01 GB
Warming up...

Running 5 prompts...
  Prompt 1: 0.8 tok/s | TTFT: 1.228s | Total: 122.07s
  Prompt 2: 0.8 tok/s | TTFT: 1.219s | Total: 125.16s
  Prompt 3: 0.8 tok/s | TTFT: 1.227s | Total: 122.43s
  Prompt 4: 0.8 tok/s | TTFT: 1.225s | Total: 122.70s
  Prompt 5: 0.8 tok/s | TTFT: 1.230s | Total: 122.35s

DeepSpeed Averages:
  Avg TTFT:        1.226s
  Avg throughput:  0.8 tokens/sec
  Avg total time:  122.94s
  Peak GPU memory: 2.46 GB


## COMPARISON TABLE

In [ ]:
print("\n" + "=" * 70)
print("RESULTS: PyTorch vs DeepSpeed ZeRO Inference (OPT-6.7B)")
print("=" * 70)
print(f"{'Metric':<25} {'PyTorch':>15} {'DeepSpeed':>15} {'Winner':>10}")
print("-" * 70)
print(f"{'Avg TTFT (s)':<25} {pytorch_avg_ttft:>15.3f} {ds_avg_ttft:>15.3f} {'PyTorch' if pytorch_avg_ttft < ds_avg_ttft else 'DeepSpeed':>10}")
print(f"{'Avg Throughput (tok/s)':<25} {pytorch_avg_throughput:>15.1f} {ds_avg_throughput:>15.1f} {'PyTorch' if pytorch_avg_throughput > ds_avg_throughput else 'DeepSpeed':>10}")
print(f"{'Avg Total Time (s)':<25} {pytorch_avg_total:>15.2f} {ds_avg_total:>15.2f} {'PyTorch' if pytorch_avg_total < ds_avg_total else 'DeepSpeed':>10}")
print(f"{'Peak GPU Memory (GB)':<25} {pytorch_peak_memory:>15.2f} {ds_peak_memory:>15.2f} {'PyTorch' if pytorch_peak_memory < ds_peak_memory else 'DeepSpeed':>10}")
print("-" * 70)
print("\nExpected outcome:")
print("  → PyTorch wins on SPEED (no CPU↔GPU data transfer overhead)")
print("  → DeepSpeed wins on MEMORY (parameters offloaded to CPU)")
print("  → This shows the tradeoff: DeepSpeed sacrifices speed for memory savings")
print("  → When the model DOESN'T fit (Exp 1), DeepSpeed is the only option!")




RESULTS: PyTorch vs DeepSpeed ZeRO Inference (OPT-6.7B)
Metric                            PyTorch       DeepSpeed     Winner
----------------------------------------------------------------------
Avg TTFT (s)                        0.025           1.226    PyTorch
Avg Throughput (tok/s)               39.8             0.8    PyTorch
Avg Total Time (s)                   2.51          122.94    PyTorch
Peak GPU Memory (GB)                13.39            2.46  DeepSpeed
----------------------------------------------------------------------

Expected outcome:
  → PyTorch wins on SPEED (no CPU↔GPU data transfer overhead)
  → DeepSpeed wins on MEMORY (parameters offloaded to CPU)
  → This shows the tradeoff: DeepSpeed sacrifices speed for memory savings
  → When the model DOESN'T fit (Exp 1), DeepSpeed is the only option!


In [ ]:

# =============================================================================
# PLOT RESULTS
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DeepSpeed ZeRO Inference vs PyTorch — OPT-6.7B on A100-40GB", fontsize=16, fontweight="bold")

colors_pytorch = "#4285F4"
colors_deepspeed = "#34A853"
width = 0.35

# --- Plot 1: Throughput per prompt ---
ax1 = axes[0, 0]
prompts_labels = [f"P{i+1}" for i in range(len(TEST_PROMPTS))]
pt_throughputs = [r["tokens_per_sec"] for r in pytorch_results]
ds_throughputs = [r["tokens_per_sec"] for r in ds_results]
x = range(len(prompts_labels))

ax1.bar([i - width/2 for i in x], pt_throughputs, width, label="PyTorch", color=colors_pytorch, alpha=0.8)
ax1.bar([i + width/2 for i in x], ds_throughputs, width, label="DeepSpeed", color=colors_deepspeed, alpha=0.8)
ax1.set_xlabel("Prompt")
ax1.set_ylabel("Tokens/sec")
ax1.set_title("Throughput per Prompt")
ax1.set_xticks(x)
ax1.set_xticklabels(prompts_labels)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# --- Plot 2: Time to First Token per prompt ---
ax2 = axes[0, 1]
pt_ttfts = [r["ttft"] for r in pytorch_results]
ds_ttfts = [r["ttft"] for r in ds_results]

ax2.bar([i - width/2 for i in x], pt_ttfts, width, label="PyTorch", color=colors_pytorch, alpha=0.8)
ax2.bar([i + width/2 for i in x], ds_ttfts, width, label="DeepSpeed", color=colors_deepspeed, alpha=0.8)
ax2.set_xlabel("Prompt")
ax2.set_ylabel("Seconds")
ax2.set_title("Time to First Token per Prompt")
ax2.set_xticks(x)
ax2.set_xticklabels(prompts_labels)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

# --- Plot 3: Average metrics comparison ---
ax3 = axes[1, 0]
metrics = ["Throughput\n(tok/s)", "TTFT\n(s)", "Total Time\n(s)"]
pt_vals = [pytorch_avg_throughput, pytorch_avg_ttft, pytorch_avg_total]
ds_vals = [ds_avg_throughput, ds_avg_ttft, ds_avg_total]
x_m = range(len(metrics))

ax3.bar([i - width/2 for i in x_m], pt_vals, width, label="PyTorch", color=colors_pytorch, alpha=0.8)
ax3.bar([i + width/2 for i in x_m], ds_vals, width, label="DeepSpeed", color=colors_deepspeed, alpha=0.8)
ax3.set_ylabel("Value")
ax3.set_title("Average Metrics Comparison")
ax3.set_xticks(x_m)
ax3.set_xticklabels(metrics)
ax3.legend()
ax3.grid(axis="y", alpha=0.3)

# --- Plot 4: Peak GPU Memory ---
ax4 = axes[1, 1]
categories = ["PyTorch", "DeepSpeed"]
memory_vals = [pytorch_peak_memory, ds_peak_memory]
bar_colors = [colors_pytorch, colors_deepspeed]

bars = ax4.bar(categories, memory_vals, color=bar_colors, alpha=0.8, width=0.5)
ax4.axhline(y=40, color="red", linestyle="--", linewidth=2, label="A100-40GB limit")
ax4.set_ylabel("GPU Memory (GB)")
ax4.set_title("Peak GPU Memory Usage")
ax4.legend()
ax4.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, memory_vals):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f} GB', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("exp2_comparison_plot.png", dpi=150, bbox_inches="tight")
print("\nPlot saved to: exp2_comparison_plot.png")

# Save raw results
results = {
    "pytorch": {
        "avg_ttft": pytorch_avg_ttft,
        "avg_throughput": pytorch_avg_throughput,
        "avg_total_time": pytorch_avg_total,
        "peak_memory_gb": pytorch_peak_memory,
    },
    "deepspeed": {
        "avg_ttft": ds_avg_ttft,
        "avg_throughput": ds_avg_throughput,
        "avg_total_time": ds_avg_total,
        "peak_memory_gb": ds_peak_memory,
    },
}
with open("exp2_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Raw results saved to: exp2_results.json")

# Cleanup
torch.distributed.destroy_process_group()
print("\nDone!")


Plot saved to: exp2_comparison_plot.png
Raw results saved to: exp2_results.json

Done!
